# 03 — Entraînement LoRA

**Début** : charge `02_preprocessing`. **Fin** : checkpoint + state.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "mlops").is_dir():
        REPO_ROOT = p
        break
sys.path.insert(0, str(REPO_ROOT))

import time

import torch
from torch.optim import AdamW
from tqdm import tqdm

from shared.paths import STEP_03_CHECKPOINT
from shared.pokemon_dataset import build_loaders_from_manifest, pick_device
from shared.sd_lora_models import DEFAULT_MODEL_ID, build_sd_lora_stack
from shared.step_artifacts import load_previous_step_output, rel_path, resolve_path, save_step_output

prev = load_previous_step_output("03_train_model")
manifest_path = resolve_path(prev["manifest_path"])
train_dataset, val_dataset, train_loader, val_loader, tokenizer = build_loaders_from_manifest(manifest_path)

device = pick_device()
model_id = DEFAULT_MODEL_ID
vae, unet, text_encoder, noise_scheduler = build_sd_lora_stack(device, model_id=model_id)
CHECKPOINT_PATH = STEP_03_CHECKPOINT
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
EPOCHS = 12
LR = 1e-5
optimizer = AdamW(unet.parameters(), lr=LR)
train_losses, val_losses, start_epoch = [], [], 0

if CHECKPOINT_PATH.exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    unet.load_state_dict(ckpt["unet_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"]
    train_losses = ckpt["train_losses"]
    val_losses = ckpt["val_losses"]
    print(f"Reprise epoch {start_epoch}")

def encode_prompt(te, input_ids, attention_mask):
    with torch.no_grad():
        return te(input_ids, attention_mask=attention_mask)[0]

if start_epoch < EPOCHS:
    for epoch in range(start_epoch, EPOCHS):
        unet.train()
        running = 0.0
        for batch in tqdm(train_loader, desc=f"Train {epoch+1}/{EPOCHS}"):
            pv = batch["pixel_values"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            with torch.no_grad():
                latents = vae.encode(pv).latent_dist.sample() * 0.18215
            enc = encode_prompt(text_encoder, ids, mask)
            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            ts = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
            pred = unet(noise_scheduler.add_noise(latents, noise, ts), ts, enc).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer.step()
            running += loss.item() * bsz
        train_losses.append(running / len(train_dataset))

        unet.eval()
        running = 0.0
        with torch.no_grad():
            for batch in val_loader:
                pv = batch["pixel_values"].to(device)
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                latents = vae.encode(pv).latent_dist.sample() * 0.18215
                enc = encode_prompt(text_encoder, ids, mask)
                noise = torch.randn_like(latents)
                bsz = latents.shape[0]
                ts = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
                pred = unet(noise_scheduler.add_noise(latents, noise, ts), ts, enc).sample
                running += torch.nn.functional.mse_loss(pred.float(), noise.float()).item() * bsz
        val_losses.append(running / len(val_dataset))
        print(f"Epoch {epoch+1}: train={train_losses[-1]:.4f} val={val_losses[-1]:.4f}")
        torch.save({
            "epoch": epoch + 1,
            "unet_state_dict": unet.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_losses": train_losses,
            "val_losses": val_losses,
            "model_id": model_id,
        }, CHECKPOINT_PATH)

save_step_output("03_train_model", {
    "manifest_path": prev["manifest_path"],
    "checkpoint_path": rel_path(CHECKPOINT_PATH),
    "model_id": model_id,
    "epochs_done": len(train_losses),
    "final_train_loss": train_losses[-1] if train_losses else None,
    "final_val_loss": val_losses[-1] if val_losses else None,
})
